# One analytical API, two execution backends

Spark should change how data moves, not the statistical question. In this
notebook we run the same A/B analysis on pandas and Spark, compare the
outputs, inspect aggregate-first tests, control caching, and finish with a
small distributed matching example.

**Requirements:** Java 8/11/17 and PySpark 3.5.1. The setup detects a Java
runtime on `PATH` and common Homebrew installations on macOS. The examples
are intentionally small; a local two-core session is enough.

## 1. Start a reproducible local session

`PYARROW_IGNORE_TIMEZONE` is set before importing PySpark to keep
pandas-on-Spark behaviour consistent. Both Spark Python variables point to
the current kernel interpreter so workers cannot accidentally use another
Python version. Few shuffle partitions avoid the overhead of Spark's
default 200 partitions on a tiny tutorial dataset.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("PYARROW_IGNORE_TIMEZONE", "1")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


def java_is_available():
    try:
        return (
            subprocess.run(
                ["java", "-version"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                check=False,
            ).returncode
            == 0
        )
    except OSError:
        return False


if not java_is_available():
    java_candidates = [
        Path("/opt/homebrew/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home"),
        Path("/usr/local/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home"),
        Path("/usr/local/opt/openjdk/libexec/openjdk.jdk/Contents/Home"),
    ]
    java_home = next(
        (path for path in java_candidates if (path / "bin/java").exists()),
        None,
    )
    if java_home is None:
        raise RuntimeError(
            "Java 8/11/17 is required. Set JAVA_HOME or add java to PATH. "
            "On macOS with Homebrew: brew install openjdk@11."
        )
    os.environ["JAVA_HOME"] = str(java_home)
    os.environ["PATH"] = f"{java_home / 'bin'}:{os.environ['PATH']}"

In [2]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession

from hypex import AATest, ABTest, Matching
from hypex.comparators import StatsZTest
from hypex.dataset import (
    Dataset,
    FeatureRole,
    InfoRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils import BackendsEnum

spark = (
    SparkSession.builder.master("local[2]")
    .appName("HypExAdvancedTutorial")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Java:", os.environ.get("JAVA_HOME", "from PATH"))

26/11/13 10:43:35 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 192.168.1.64 instead (on interface en0)
26/11/13 10:43:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/13 10:43:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.1
Java: /opt/homebrew/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home


## 2. Construct both datasets from the same rows

The roles are identical. Only `backend` and `session` differ. `data`
exposes the underlying pandas or Spark DataFrame, while `raw_data` is the
pandas-on-Spark representation on the Spark backend; most user code should
stay with the backend-agnostic `Dataset` API.

In [3]:
rng = np.random.default_rng(77)
n_users = 2_000
pre_revenue = rng.gamma(4, 20, n_users)
treat = rng.binomial(1, 0.5, n_users)
revenue = pre_revenue + 6 * treat + rng.normal(0, 12, n_users)
converted = rng.binomial(1, 1 / (1 + np.exp(-(-2.5 + 0.02 * revenue))))
channel = np.where(
    treat == 1,
    rng.choice(["organic", "paid"], n_users, p=[0.6, 0.4]),
    rng.choice(["organic", "paid"], n_users, p=[0.5, 0.5]),
)
backend_df = pd.DataFrame(
    {
        "user_id": np.arange(n_users),
        "treat": treat,
        "revenue": revenue,
        "converted": converted,
        "channel": channel,
        "age": rng.normal(38, 10, n_users),
        "pre_spend": pre_revenue,
    }
)
roles = {
    "user_id": InfoRole(int),
    "treat": TreatmentRole(int),
    "revenue": TargetRole(float),
    "converted": TargetRole(int),
    "channel": TargetRole(str),
    "age": FeatureRole(float),
    "pre_spend": FeatureRole(float),
}
pandas_data = Dataset(roles=roles, data=backend_df)
spark_data = Dataset(
    roles=roles,
    data=backend_df,
    backend=BackendsEnum.spark,
    session=spark,
)
print("pandas:", type(pandas_data.data).__module__)
print("Spark:", type(spark_data.data).__module__)
print("Backend enum:", spark_data.backend_type)

pandas: pandas.core.frame
Spark: pyspark.sql.dataframe
Backend enum: BackendsEnum.spark


## 3. Run the same A/B pipeline

On pandas, t-, KS- and U-tests operate on raw columns through SciPy. On
Spark, their registered implementations aggregate first. T-test moments and
chi-square counts support equivalent calculations; KS and Mann–Whitney use
histogram approximations, so small numerical differences are expected.

In [4]:
def make_ab_test():
    return ABTest(
        additional_tests=["t-test", "ks-test", "u-test", "chi2-test"],
        multitest_method="holm",
    )


pandas_result = make_ab_test().execute(pandas_data)
spark_result = make_ab_test().execute(spark_data)
spark_result.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value,UTest pass,UTest p-value,Chi2Test pass,Chi2Test p-value
0,revenue,1,79.388506,86.256069,6.867563,8.650576,3.266323,10.468802,OK,0.000191,OK,0.000064,OK,0.000030,NaN,NaN
1,converted,1,0.331373,0.323469,-0.007903,-2.384978,-0.049062,0.033255,NOT OK,0.706692,NOT OK,1.000000,NOT OK,0.706662,NaN,NaN
2,channel,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OK,5.889447e-10


In [5]:
comparison_columns = [
    "feature",
    "group",
    "difference",
    "TTest p-value",
    "KSTest p-value",
    "UTest p-value",
    "Chi2Test p-value",
]
pandas_summary = pandas_result.summary.raw_data[comparison_columns]
spark_summary = spark_result.summary.raw_data[comparison_columns]
parity = pandas_summary.merge(
    spark_summary,
    on=["feature", "group"],
    suffixes=(" pandas", " spark"),
)
parity.round(5)

,feature,group,difference pandas,TTest p-value pandas,KSTest p-value pandas,UTest p-value pandas,Chi2Test p-value pandas,difference spark,TTest p-value spark,KSTest p-value spark,UTest p-value spark,Chi2Test p-value spark
0,revenue,1,6.86756,0.00019,0.00005,0.00003,NaN,6.86756,0.00019,0.00006,0.00003,NaN
1,converted,1,-0.00790,0.70669,1.00000,0.70666,NaN,-0.00790,0.70669,1.00000,0.70666,NaN
2,channel,1,NaN,NaN,NaN,NaN,0.00001,NaN,NaN,NaN,NaN,0.0


The effect estimates should agree up to floating-point aggregation order.
For approximate tests, assess the conclusion and the magnitude of the
difference rather than demanding bit-for-bit equality. Histogram-based KS
and U tests use `n_bins`; extreme outliers can stretch the bins and reduce
approximation quality.

## 4. Aggregate-first statistics as standalone calculators

`StatsZTest` needs only counts and sums to compare two proportions:

$$z=rac{\hat p_1-\hat p_2}
{\sqrt{\hat p(1-\hat p)(1/n_1+1/n_2)}}.$$

On Spark this means a distributed aggregation followed by a small driver-
side calculation, not a transfer of every raw outcome to SciPy.

In [6]:
ztest = StatsZTest.calc(
    target_fields_data=spark_data[["converted"]],
    group_field_data=spark_data[["treat"]],
    reliability=0.05,
)
ztest[0].raw_data

,p-value,statistic,pass
0,0.706597,0.376485,0.0


## 5. Cache deliberately, then release

Repeated splits and aggregations can otherwise recompute the same Spark
lineage. Shells auto-persist an uncached Spark input for the duration of
`execute`, then release only what they cached themselves. A dataset you
explicitly persist remains cached after the shell returns.

In [7]:
spark_data.persist(storage_level="MEMORY_AND_DISK", action="count")
print(
    {
        "is_persisted": spark_data.is_persisted,
        "storage_level": spark_data.get_storage_level(),
    }
)
cached_result = ABTest(additional_tests="t-test").execute(spark_data)
print("Still cached after shell execution:", spark_data.is_persisted)
spark_data.unpersist()
print("After explicit release:", spark_data.is_persisted)

{'is_persisted': True, 'storage_level': 'MEMORY_AND_DISK'}


Still cached after shell execution: True
After explicit release: False


## 6. A/A and matching also keep the same shell API

The A/A example uses only ten iterations to keep the tutorial quick. Use
hundreds or thousands for a serious false-positive-rate study. Matching
still uses FAISS for nearest-neighbour search: Spark distributes the data
preparation, but this is not a promise that every model operation becomes
distributed. Wide `full_data` extraction is left off because joins and
checkpoints are expensive.

In [8]:
spark_aa = AATest(n_iterations=10, stratification=False).execute(spark_data)
spark_aa.aa_score

 10%|█         | 1/10 [00:02<00:19,  2.15s/it]

 20%|██        | 2/10 [00:03<00:15,  1.94s/it]

 30%|███       | 3/10 [00:05<00:12,  1.74s/it]

 40%|████      | 4/10 [00:06<00:09,  1.65s/it]

 50%|█████     | 5/10 [00:08<00:08,  1.61s/it]

 60%|██████    | 6/10 [00:09<00:06,  1.53s/it]

... (2 outputs left out)

 90%|█████████ | 9/10 [00:13<00:01,  1.34s/it]

100%|██████████| 10/10 [00:14<00:00,  1.28s/it]

100%|██████████| 10/10 [00:14<00:00,  1.49s/it]

,score,pass
revenue TTest test_1,0.95,True
converted TTest test_1,0.95,True
age TTest test_1,0.95,True
pre_spend TTest test_1,0.95,True
revenue KSTest test_1,0.95,True
converted KSTest test_1,0.95,True
age KSTest test_1,0.95,True
pre_spend KSTest test_1,0.95,True
channel Chi2Test test_1,0.95,True


In [9]:
matching_roles = {
    "user_id": InfoRole(int),
    "treat": TreatmentRole(int),
    "revenue": TargetRole(float),
    "age": FeatureRole(float),
    "pre_spend": FeatureRole(float),
    "channel": FeatureRole(str),
}
matching_columns = [
    "user_id",
    "treat",
    "revenue",
    "age",
    "pre_spend",
    "channel",
]
spark_matching_data = Dataset(
    roles=matching_roles,
    data=backend_df[matching_columns],
    backend=BackendsEnum.spark,
    session=spark,
)
spark_matching = Matching(
    distance="mahalanobis",
    quality_tests=[],
    extract_full_data=False,
).execute(spark_matching_data)
spark_matching.summary

/Users/danilsamsutdinov/HypEx/.worktrees/advanced-tutorials/hypex/dataset/backends/spark_backend.py:180: UserWarning: SparkContext checkpoint directory is not set. Falling back to local_checkpoint, which is NOT fault-tolerant and may fail if an executor is lost. Please set sc.setCheckpointDir('hdfs://...') for large jobs.
  warnings.warn(


/Users/danilsamsutdinov/HypEx/.worktrees/advanced-tutorials/hypex/dataset/backends/spark_backend.py:180: UserWarning: SparkContext checkpoint directory is not set. Falling back to local_checkpoint, which is NOT fault-tolerant and may fail if an executor is lost. Please set sc.setCheckpointDir('hdfs://...') for large jobs.
  warnings.warn(


/Users/danilsamsutdinov/HypEx/.worktrees/advanced-tutorials/hypex/dataset/backends/spark_backend.py:180: UserWarning: SparkContext checkpoint directory is not set. Falling back to local_checkpoint, which is NOT fault-tolerant and may fail if an executor is lost. Please set sc.setCheckpointDir('hdfs://...') for large jobs.
  warnings.warn(


26/11/13 10:44:08 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/13 10:44:08 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


/Users/danilsamsutdinov/HypEx/.worktrees/advanced-tutorials/hypex/dataset/backends/spark_backend.py:180: UserWarning: SparkContext checkpoint directory is not set. Falling back to local_checkpoint, which is NOT fault-tolerant and may fail if an executor is lost. Please set sc.setCheckpointDir('hdfs://...') for large jobs.
  warnings.warn(


,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,7.1095,0.9596,0.0,5.2286,8.9904
ATC,7.0903,0.9444,0.0,5.2393,8.9414
ATE,7.0997,0.8537,0.0,5.4265,8.7730


In [10]:
spark.stop()

## Backend checklist

- Keep roles and experiment definitions backend-independent.
- Expect exact parity for moment-based tests within numerical tolerance,
  and approximate parity for histogram-based KS/U tests.
- Reduce `spark.sql.shuffle.partitions` for small local jobs; tune it for
  data size on a cluster.
- Persist data reused across stages, but unpersist it when finished.
- Remember that result tables are intentionally small pandas-backed
  `SmallDataset` objects even when raw data stays distributed.
- Test the exact executor combination on Spark before scaling up; optional
  model and reporting operations have different cost profiles.

Return to the [advanced tutorials index](README.md).